### Tools

Models can request to call tools that perform tasks such as fetching data from a database, searching the web, and running code.

Tools are pairings of:

#### 1. Tool Schema

The schema defines the tool's interface, including:
- **Name:** The name of the tool.
- **Description:** Explains what the tool does.
- **Arguments:** Defines the inputs the tool accepts, often using a JSON schema.

#### 2. Function or Coroutine

A function or coroutine executes the actual task when the tool is called.

- **Function:** Executes a task, usually synchronously.
- **Coroutine:** Executes an asynchronous task, allowing other operations to proceed while waiting for the result.


In [1]:

import os
from dotenv import load_dotenv
from langchain_groq import ChatGroq

load_dotenv()

api_key = os.getenv("GROQ_API_KEY")

if not api_key:
    raise ValueError("GROQ_API_KEY is missing. Check your .env file.")

model = ChatGroq(
    model="openai/gpt-oss-120b",
    api_key=api_key,
    reasoning_format="hidden",
)


In [3]:
from langchain.tools import tool

@tool
def get_weather(location: str) -> str:
    """Get the current weather for a given location."""
    # Replace this with actual weather fetching logic
    return f"The weather in {location} is sunny."

model_with_tools=model.bind_tools([get_weather])


In [6]:
response = model_with_tools.invoke("What's the weather in New York?")
print(response)
for tool_call in response.tool_calls:
    print(f"Tool: {tool_call['name']}")
    print(f"Arguments: {tool_call['args']}")


content='' additional_kwargs={'tool_calls': [{'id': 'fc_fae9dbe1-6b4f-4071-a66f-10b09fea0c28', 'function': {'arguments': '{"location":"New York"}', 'name': 'get_weather'}, 'type': 'function'}]} response_metadata={'token_usage': {'completion_tokens': 45, 'prompt_tokens': 129, 'total_tokens': 174, 'completion_time': 0.094890211, 'completion_tokens_details': {'reasoning_tokens': 17}, 'prompt_time': 0.005073385, 'prompt_tokens_details': None, 'queue_time': 0.048616692, 'total_time': 0.099963596}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_8ea50c0161', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'} id='lc_run--01a16735-2387-7113-a648-1509671075ad-0' tool_calls=[{'name': 'get_weather', 'args': {'location': 'New York'}, 'id': 'fc_fae9dbe1-6b4f-4071-a66f-10b09fea0c28', 'type': 'tool_call'}] invalid_tool_calls=[] usage_metadata={'input_tokens': 129, 'output_tokens': 45, 'total_tokens': 174, 'output_token_details': {'re

## Tool execution loop

In [7]:
# Step1 : Model generates tool calls
messages = [{"role": "user", "content": "What's the weather in New York?"}]
ai_msg = model_with_tools.invoke(messages)
messages.append(ai_msg)

#Step 2 : Execute tools and collect results
for tool_call in ai_msg.tool_calls:
    tool_result = get_weather.invoke(tool_call)
    messages.append(tool_result)

final_response = model_with_tools.invoke(messages)
print(final_response.text)
    


The current weather in New York is sunny. 🌞
